In [ ]:
import logging
import time
from corriscope.fpga_array import FPGAArray
from corriscope.hardware.crs import CRS
import numpy as np
%matplotlib ipympl
import matplotlib.pyplot as plt

# Must use %matplotlib widget with jupyter-lab. Needs 'pip install ipympl'
# jupyter-notebook needs %matplotlib notebook
plt.rcParams['figure.figsize'] = (9.5, 6)

log=logging.getLogger()
log.setLevel(logging.DEBUG)


In [ ]:
# Backplane lane connectivity map
# tx_to_rx_lane_map = { # (Tx_slot, Tx_lane):(Rx_slot, Rx_lane), slots are 1-based
#  (1,0): (1,0),    (1, 1): (4, 3), (1, 2): (3, 3), (1, 3): (2, 3),
#  (2,0): (2,0),    (2, 1): (1, 3), (2, 2): (3, 2), (2, 3): (4, 2),
#  (3,0):(3,0),     (3, 1): (4, 1), (3, 2): (2, 2), (3, 3): (1, 2),
#  (4,0):(4,0),     (4, 1): (3, 1), (4, 2): (2, 1), (4, 3): (1, 1)}
# rx_to_tx_lane_map = {rx:tx for tx,rx in tx_to_rx_lane_map.items()}

# Bin ordering
f_nat = np.arange(8192)
f_rev = (f_nat>>11) | ((f_nat & 0b11111111111) << 2)
frev2 = (f_rev & 0b1111111111100) | ((f_rev + np.arange(8192)) & 0b11)


In [ ]:
frev2[:16]

In [ ]:
[[0, 1, 2, 3, 4, ... 2047],
 [2048, ... 4095],
 [4096 , ...]
 [6144, ...]]
 
v = [0, 2048, 4096, 6144, 4, ...]
v[f_rev] = [0, 1, 2 , 3 ...]
bin0 v[0], bin1 v[4] bin 3 = v[8] ...

 

In [ ]:
# Compute bin numbers at the output of the FFT, including bin rotation. 
# We split the data into 4 lanes since the FFT outputs 4 numbers per clock. 
fft_bins = np.array([[2048*((lane - (clk & 0b11)) & 0b11) + clk for clk in range(2048)] for lane in range(4)]) 

# Compute the bin numbers in each lane of 1st corner-turn. CT1 has one output lane for each FFT output lane. 
# ct1_bins: (lane, binid) = bin, ct1_bins.shape = (4,2048)
ct1_bins = fft_bins 

# Cull bins 
# The 2nd CT can transfer 25 Gbps/lane, but our data has 25.6 Gbps/lane. 
# We need to drop some bins in order to fit the pipe. Here we remove 256 bins total, i.e. 64 bins per lane.
# The culling takes into account the bin rotation and remove only the first 256 bins (first 50 MHz) of the spectrum
# Note: The FFT bin rotation was added to ensure the culling of low-frequency bins affects all lanes equally. 
#       Otherwise, with the bitgrowth FFT without unscranbler,  all low frequency bins would have been in lane 0.
# ---------
# ct2_bins: (lane, binid) = bin, ct2_bins.shape = (4,1984), 1984=2048-256/4
ct2_bins = np.array([[b for clk,b in enumerate(ct1_lane) if not (clk < 256 and clk & 0b11 == lane)] for lane,ct1_lane in enumerate(ct1_bins)]) 
# that are sent to each slot,  

# Remap & shuffle
# ---------------
# We do nothing here because we assume the lane map is configured to cancel the shuffle reordering
# i.e. to send bins from lane 0 to slot 0, bins from lane 1 to slot 1 etc...
# ct2_bins: (slot, binid) = bin, ct2_bins.shape = (4,1984)

# Channel remapping
# -----------------
# Now, although the right bins end up in the right slots, the backplane wiring has scrambled the 
# order of the channels, i.e channels 0-8, 9-15, 16-23, 24-31  are not necessarily in Rx lane 0, 1, 2 and 3.
# The channel scrambling is slot-dependent.  
# chan_map := (slot, chan_pos) = real_chan, chan_map.shape = (4, NCHAN)
# rx_to_tx = CRS.RX_TO_TX_LANE_MAP # Get the backplane lane map
# chan_map = np.array([[c + 8*rx_to_tx[(rx_slot, rx_lane)][0] for rx_lane in range(4) for c in range(8)] for rx_slot in range(4)])

# This computes the scrambled FFT outputs with bin rotation, 1st-level corner-turn, and bin culling. 
# bins_culled = np.array([[2048*((lane - (clk & 0b11)) & 0b11) + clk for clk in range(2048) if not (clk < 256 and clk & 0b11 == lane)] for lane in range(4)]) 

# Apply correlator bin decimation
# corr_bins := (slot, bin_id) = bin   corr_bins.shape=(4, 496) (496 = 1984/4)
corr_bins = ct2_bins[:, 3::4]

# Compute (i,j) product indices coming out of the correlator on each clock for a single bin 
# The map reflects the way the correlator performs the products 
# prod_map := (clk, prod) = (i, j)   prod_map.shape=(4, 132, (2))
N=32
NCLK=4
prod_map=np.array([[(i, (j+i) % N) if i < N-j else ((j+i) % N, i-1) for i in range(N+1) for j in range(N//2//NCLK*clk, N//2//NCLK*(clk+1)) ] for clk in range(NCLK) ])

# Compute (i.j) product indices for each slot, taking account the slot-dependent channel scrambling 
# prod_remap := (slot, clk, prod) = (i,j)  prod_remap.shape = (4, 4, 132, (2))
# prod_remap = np.array([slot_chan_map[prod_map] for slot_chan_map in chan_map])
# prod_remap = np.array([slot_chan_map[prod_map] for slot_chan_map in chan_map])
# prod_remap = np.array([prod_map for slot_chan_map in chan_map])

# We now have all the elements to create a map that described the bin number  and (i,j) product indices 
# for each of the values received in the packet receiver.
# The receiver stores data as (slot_id, clkbin_id, prod) .shape=(4,496,132). 
# The map profides a (bin, i, j) tuple for each of the element of that array 
# obsolete: nmap := (slot_id, clkbin_id, prod) = (bin, i, j) nmap.shape = (4, 1984, 132, (3)), 1984=4 clk * 496 bins, bin=0-8191, i,j=0-NCHAN-1
# obsolete: nmap := (corr_id, bin_id) = (bin, prod), shape=(4, 2048)
# nmap := (slot_id, clkbin_id, prod) = (bin, prod) nmap.shape = (4, 1984, 132, (3)), 1984=4 clk * 496 bins, bin=0-8191, i,j=0-NCHAN-1
nmap =  np.array([[[(bin_, clk*132+prod) for prod in range(132)] for bin_ in corr_bins[slot] for clk in range(NCLK)] for slot in range(4)])



In [ ]:
nmap[0,4,0]

In [ ]:
fft_bins.shape

In [ ]:
fft_bins[:,:10]

In [ ]:
ct2_bins.shape

In [ ]:
ct2_bins[:,:16]

In [ ]:
corr_bins[:,:10]

In [ ]:
# bins that are sacrificed for readout
corr_bins[:,0]

In [ ]:
prod_map

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Bins processed by each slot after correlator decimation')
ax.set_xlabel('Index')
ax.set_ylabel('Bin number')
for slot in range(4): 
    ax.plot(np.arange(1984/4), corr_bins[slot], '.', label=f'Slot {slot}')
ax.legend()

In [ ]:
prod_map.shape

In [ ]:
# full crate: 48 38 47 36   51 44 37 42   NC NC NC 35 39 49 52 50
# 
# ca = FPGAArray('crs 35 49 51', mode='corr32', stderr_log_level='DEBUG')
# ca = FPGAArray('crs 47:1 38:2 48:3 36:4', mode='corr32', sync_method='irigb', sync_source='bp_trig', stderr_log_level='warn')
# ca = FPGAArray('crs 48:1 38:2', mode='corr32', sync_method='irigb', sync_source='bp_trig', stderr_log_level='warn', init=2)
# hwm = 'crs 48:1 38:2 47:3 36:4    51:5 44:6 37:7 42:8                35:12 39:13 49:14 52:15 50:16'
# hwm = 'crs 48:1'
# hwm = 'crs 48:1 38:2 47:3 36:4    51:5 44:6 37:7 42:8                      39:13 49:14 52:15 50:16'
# hwm = 'crs 48:1 38:2 47:3 36:4'
hwm = 'crs 37:1 39:2 48:3 11:4'
hwm = 'crs 50:1'
ca = FPGAArray(hwm, mode='corr32', sync_method='local', sync_source='bp_trig', sync_master=1, sync_master_source='irigb_gen', sync_master_output=1, stdout_log_level='warn')

In [ ]:
# bp = [i.CT.BPLINKS for  i in ca.ib]
# y=[y for  i in ca.ib for y in i.CT.BPLINKS.gty]
i = ca.ib[0]
scaler = i.chan[0].SCALER
funcgen = i.chan[0].FUNCGEN
fft = i.chan[0].FFT
ucap = i.UCAP
ucorr = i.CORR
Fs = i._sampling_frequency
# await i.set_local_data_port_number_async(41234)  # use a separate UDP data destination port in case someone else is using it
# r = i.get_data_receiver()
# cr = i.get_corr_receiver()
# cr.ignore_packet_size = True
gpio=i.GPIO


In [ ]:
i = ca.ib[0]
r = i.get_data_receiver()

In [ ]:
# i.set_channelizer(data_source='a', a=1, fft_bypass=True, scaler_bypass=False); i.set_gains(1<<16)
# i.set_channelizer(data_source='sin', freq=1, fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False); i.set_gains(40*2**10)
# i.set_channelizer(data_source='ramp', fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False); i.set_gains(60*2**10)
i.set_channelizer(data_source='adc', funcgen_left_shift=15, funcgen_right_shift=0, 
                  fft_bypass=False, 
                  scaler_bypass=False, offset_binary_encoding=False, scaler_rounding_mode=2, 
                  gains=1*2**32)
# funcgen.SHIFT=0
# ucap.OUTPUT_SOURCE_SEL = 0
# scaler.ROUNDING_MODE = 2
# scaler.CAP_DATA_TYPE=2 # 0=main output, 1=raw fft, 2=post-gain FFT, 3=4+4 bit values, 4: Raw FFT even bins, 5: Raw FFt odd bins, 6-7: zero
i.start_data_capture(.1, source='scaler', mode=3, channels=[0], data_type='fft_gain', select=True)

In [ ]:

# ucap.MODE = 3  # single-channel mode, 16 frames/channel
t,d,c = r.read_raw_frames(ncap=100, split=True, format='16+16', verbose=1)

In [ ]:
c

In [ ]:
d.shape

In [ ]:
dd=d[0, :, frev2]
dd.shape

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Output of the ADC, terminated amplifier chain')
ax.set_xlabel('Sample #')
ax.set_ylabel('ADC output')
ax.plot(abs(dd).mean(axis=-1))
# ax.plot(abs(d[0,0,frev2]))
d.shape

In [ ]:
# Create correlator receiver, making sure they all operate on the same socket
for i in ca.ib:
    i.fpga._data_socket = None
    sock=i.get_data_socket(41000)
    print(f'{i.hostname} {sock}')
cr = ca.ib[0].CORR.get_data_receiver(sock=sock, NBOARDS=4)


In [ ]:
#Get corr output
for i in ca.ib:
    # i.start_correlator()
    # i.set_channelizer(data_source='ab',a=1<<10, b=0<<10, fft_bypass=True, scaler_bypass=True);

    
    # i.set_channelizer(fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False, gains=1<<26);
    # i.set_channelizer(data_source='ab', a=0, b=0);

    
    # i.set_channelizer(data_source='ab', a=1, b=0, channels=[0]);

    data = np.zeros(8192, dtype=np.complex64)
    data[[6145, 4098, 2051, 6149, 4102, 2055, 6153, 4106, 2059, 6157, 4110,
        2063, 6161, 4114, 2067, 6165]]=1
    i.set_channelizer(data_source='buffer', buffer=np.ravel((data.real, data.imag), order='F'), channels=[0]);

    # i.set_channelizer(data_source='adc', fft_bypass=False, scaler_bypass=False, offset_binary_encoding=False); i.set_gains(1*2**55)
    # i.set_channelizer(data_source='adc', fft_bypass=False, scaler_bypass=False, use_offset_binary=False); i.set_gains(1<<47)

    i.set_channelizer(data_source='adc', fft_bypass=False, scaler_bypass=False, offset_binary_encoding=False,
                     gains = 1<<33); 

    # i.set_channelizer(data_source='sin', freq=263, fft_bypass=False, scaler_bypass=False, offset_binary_encoding=False, gains = 1<<29); 
    
    # i.set_gains(10*2**28) # was 29
    # i.set_channelizer(data_source='ab', a=20, b=0)
    # i.set_channelizer(data_source='noise', fft_bypass=True, scaler_bypass=False); i.set_gains(1.5*2**12)
    
    i.UCAP.OUTPUT_SOURCE_SEL = 1 # 0= raw data, 1=CORR
    i.CORR.NO_ACCUM = 0
    i.CORR.AUTOCORR_ONLY = 0
    i.CORR.INTEGRATION_PERIOD = 16384-1
    # i.start_data_capture(1)

await ca.sync_async()

In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.001, soft_integ_period=1, number_of_results=3, verbose=0)
print(np.sum(cc0, axis=-1))
print(f'Bins sacrified for readout: {corr_bins[:,0]}') 
# remapping data 
a=np.full(8192, np.NaN, dtype=np.complex128)
for cid, d0 in enumerate(nmap): # cid=slot, d0=[bix,pix] = (bin,prod)
    for bid, d1 in enumerate(d0): # bix =bix, d1=[pix] = (bin, prod)
        for pid, (bin, prod) in enumerate(d1):
            if prod == 0:
                assert np.isnan(a[bin]), "reassigning a value"
                a[bin] = dd0[0,cid,bid,pid]
            # print(f'a[{bin}] = dd0[0, {cid:1d},{bid:4d},{pid:3d}] = {a[bin]}')
# a=np.array([])
fig, ax = plt.subplots()
ax.set_title('Autocorr CH0')
ax.plot(abs(a),'.', ms=1)
# bb= corr_bins[:,0] # bad bins
# ax.plot(bb, abs(a[bb]), 'ro')

In [ ]:
dd0.shape

In [ ]:
(32+1)*32/2*5+10

In [ ]:
# a1+1j)*(1-1j)*16384

In [ ]:
# plt.rcParams['axes.titlepad']=0
# plt.rcParams['figure.autolayout']=True
# plt.figure('a')
fig, ax = plt.subplots()
ax.set_title('Autocorr CH0')
# plt.subplots_adjust(top=.99)
# plt.clf()
# ax.plot(abs(a))
# for p in range(32*33//2*0+1):
#     mask = ~np.isnan(a[p])
#     bin=np.arange(len(a[p]))
ax.plot(abs(a),'.', ms=1)
# for c in range(4):
#     ax.plot(np.abs(dd0[1,c,:,0]),'.', ms=1)
# nan=np.nan
# ax.plot([1,2,3,nan,4],'.')
# plt.show()
# fig.tight_layout()

In [ ]:
# Plot the number of frames for each bin
c= cc0.reshape(cc0.shape[0], cc0.shape[1], -1, NCLK).sum(axis=-1)
fig, ax = plt.subplots()
ax.set_title('Number of packets received per bin')
ax.set_xlabel('Bin #')
ax.set_ylabel('# of packets')
for i in range(len(cc0[1:2])):
    for slot in range(4):
        b = np.full(8192, np.NAN) # 
        b[nmap[slot,::NCLK,0,0]] =  c[i,slot,:496]
        # ax.plot(range(2048), cc0[i,slot]*0.1+slot+0.01*i, label=f'slot {slot}')
        ax.plot(np.arange(8192)*3200/16384, b, '.', label=f'Capture {i} Slot {slot}')
ax.legend()

In [ ]:
list(zip(*np.where(dd0[1,:,:,0])))

In [ ]:
s,b,p = np.where(dd0[1,:,:,:])

In [ ]:
list(zip(s,b,p))

In [ ]:
nmap[s,b,0]

In [ ]:
corr_bins[:,0]

d=dd0[0].reshape(4, -1).reshape(4,2048//4,-1)

In [ ]:
d.shape

In [ ]:
corr_bins[1,0]

In [ ]:
nmap[1,1,0]

In [ ]:
s=({bin:(cid, bid, pid) for cid,corrbin in enumerate(nmap) 
                        for bid,binprod in enumerate(corrbin) 
                        for pid,(bin, i, j) in enumerate(binprod) if i==0 and j==0})
# ss=sorted(s,key=lambda x:x[1][0])
# ss
s
v=[dd0[0,s[bin][0],s[bin][1],s[bin][2]] for bin in range(8192) if bin in s]
vix=np.array([(s[bin][0],s[bin][1],s[bin][2]) for bin in range(8192) if bin in s])
b=np.full(8192,np.NaN, dtype=np.complex128)
# for cid,corrbin in enumerate(nmap):
#     for bid,binprod in enumerate(corrbin) :
#         for pid,(bin, i, j) in enumerate(binprod):
#             if i==0 and j==0:
#                 b[bin] = dd0[0, cid, bid, pid]
# b[nmap[...,0]] = dd0[0,:,:1984,:]
b = dd0[0,vix[:,0], vix[:,1], vix[:,2]]
ix = np.full((4,2048,132), 0)
for bin, (c,b,p) in s.items():
    ix[c,b,p] = bin

In [ ]:
ix[0,4,0]

In [ ]:
b=np.full(8192,np.NaN, dtype=np.complex128)
b[ix]=dd0[0]

In [ ]:
plt.figure()
ax=plt.subplot(111)
# ax.plot(abs(a[:,0,0]))
ax.plot(abs(b),'.')
plt.show()

In [ ]:
[ pid==0 for bin,(cid,bid,pid) in s.items()]

In [ ]:
s={tuple(bij) for bij in nmap.reshape((-1,3))}
len(s)

In [ ]:
32*33/2*1984

In [ ]:
plt.figure(0)
ax=plt.subplot(111)
# ax.plot(abs(a[:,0,0]))
ax.plot(abs(dd0[0, :, : ,0].flatten()))
plt.show()

In [ ]:
ac={}
for i in range(-7, 8):
    for j in range(-7,8):
        ac.setdefault(int(abs((i+1j*j)*(i-1j*j))), []).append((i,j))
ac


In [ ]:
cc0[0]

In [ ]:
cc0.shape

In [ ]:
fig, ax = plt.subplots()
for corr in range(4):
    ax.plot(range(2048), cc0[4,corr]*0.1+corr)


In [ ]:
plt.figure(2)
plt.clf()
ax=plt.subplot(111)
for corr in range(4):
    ax.plot(range(2048//4), abs(dd0[0,corr,::4,0])*0.1+corr)
plt.show()

In [ ]:
cr.firmware_integ_period

In [ ]:
1984*4

In [ ]:
# cc0 is (integ, corrid, binid, prod) (integ, 4, 2048)
np.where(cc0[0])

In [ ]:
i.fpga._data_socket

In [ ]:
for i in ca.ib:
    print(f'{i._data_socket} {i._data_socket.getsockname()}')

In [ ]:
dd0.shape, cc0.shape, ss0.shape

In [ ]:
cc0.sum(axis=(1,2))

In [ ]:
d=dd0[0]
d

In [ ]:
i=ca.ib[0]
i.CORR.BIN_DECIMATION_FACTOR
cr.NCHAN

In [ ]:
32*33/2/4*5

In [ ]:
ca.ib._data_socket

In [ ]:
import __main__

In [ ]:
__main__.__opened_sockets__

In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.0001, soft_integ_period=1, number_of_results=1, align=True, verbose=1)

In [ ]:
dd0[0,:,0]

In [ ]:
i=ca.ib[1]

In [ ]:
i.start_data_capture(1)

In [ ]:
for i in ca.ib:
    i.CORR.CORR_ID = 2*(i.slot-1)
    print(f'{i.hostname} : CORR ID = {i.CORR.CORR_ID}')

In [ ]:
ca.ib[2].hostname

In [ ]:
await ca.sync_async()

In [ ]:
i = ca.ib[0]
ct2=i.CT.CT2

In [ ]:
await ca.sync_async()

In [ ]:
await ca.sync_async()

In [ ]:
for i in ca.ib:
    ct2=i.CT.CT2
    for lane in range(4):
        rx = (i.slot, lane)
        tx = rx_to_tx_lane_map[rx]
        ct2.ALIGN_MON_LANE=lane
        ct2.ALIGN_MON_SOURCE=4  # 3= FIFO occupancy, 4=lane delay,
        ct2.ALIGN_MON_RESET=1
        ct2.ALIGN_MON_RESET=0
        time.sleep(0.1)
        print(f'{tx} -> {rx}: lane {lane}: delay={ct2.ALIGN_MON_WORD:08d} clocks')

In [ ]:
set(ct2.ALIGN_MON_WORD for _ in range(10))

In [ ]:
ct2.ALIGN_SOF_WINDOW

In [ ]:
ct2.read_control(0, length=1, type=int)

In [ ]:
(i.CORR.IN_FRAME_CTR, i.CORR.OUT_FRAME_CTR)

In [ ]:
ucorr.OVERRUN

In [ ]:
ucorr.NCORR

In [ ]:
ucorr.NCHAN

In [ ]:
ucorr.NO_ACCUM=1

In [ ]:
ucorr.SOFT_RESET=0

In [ ]:
ucorr.INTEGRATION_PERIOD

In [ ]:
i.GPIO.CORR_RESET=0

In [ ]:
32*33/2*5+10

In [ ]:
ucorr.AUTOCORR_ONLY=0

In [ ]:
32*5+10

In [ ]:
64*65/2*5

In [ ]:
660/5

In [ ]:
32*33/2/4*5+10

In [ ]:
32*33/2/4

In [ ]:
8*9/2

In [ ]:
N=8    
prod_map=[(i, (j+i) % N) if i < N-j else ((j+i) % N, i-1) for i in range(N+1) for j in range(N//2)]
prod_map

In [ ]:
ax = plt.subplot(111)
mat=np.zeros((N,N))
mat[pmap[:,0], pmap[:,1]]=1
ax.matshow(mat)
plt.show()

In [ ]:
prod_map.shape

In [ ]:
prod_remap.shape   # (slot, clk, prod) = (i, j)

In [ ]:
f_nat = np.arange(8192)
f_rev = (f_nat>>11) | ((f_nat & 0b11111111111) << 2)
frev2 = (f_rev & 0b1111111111100) | ((f_rev + np.arange(8192)) & 0b11)


In [ ]:
((f_nat + (f_nat >> 2)) & 0b11)[:100]

In [ ]:
# bin = (((f_nat - (f_nat>>2)) << 11) | (f_nat >> 2)) & 0x1FFF
# Compute bin numbers at the output of the FFT, including bin rotation. 
# We split the data into 4 lanes since the FFT outputs 4 numbers per clock. 
fft_bins = np.array([[2048*((lane - (clk & 0b11)) & 0b11) + clk for clk in range(2048)] for lane in range(4)]) 

# Compute the bin numbers in each lane of 1st corner-turn. CT1 has one output lane for each FFT output lane. 
# ct1_bins: (lane, binid) = bin, ct1_bins.shape = (4,2048)
ct1_bins = fft_bins 

# Cull bins 
# The 2nd CT can transfer 25 Gbps/lane, but our data has 25.6 Gbps/lane. 
# We need to drop some bins in order to fit the pipe. Here we remove 256 bins total, i.e. 64 bins per lane.
# The culling takes into account the bin rotation and remove only the first 256 bins (first 50 MHz) of the spectrum
# In fact, the bin rotation is essential to ensure the culling affects all lanes equally.
# ---------
# ct2_bins: (lane, binid) = bin, ct2_bins.shape = (4,1984), 1984=2048-256/4
ct2_bins = np.array([[b for clk,b in enumerate(ct1_lane) if not (clk < 256 and clk & 0b11 == lane)] for lane,ct1_lane in enumerate(ct1_bins)]) 
# that are sent to each slot,  

# Remap & shuffle
# ---------------
# We do nothing here because we assume the lane map is configured to cancel the shuffle reordering
# i.e. to send bins from lane 0 to slot 0, bins from lane 1 to slot 1 etc...
# ct2_bins: (slot, binid) = bin, ct2_bins.shape = (4,1984)

# Channel remapping
# -----------------
# Now, although the right bins end up in the right slots, the backplane wiring has scrambled the 
# order of the channels, i.e channels 0-8, 9-15, 16-23, 24-31  are not necessarily in Rx lane 0, 1, 2 and 3.
# The channel scrambling is slot-dependent.  
# chan_map := (slot, chan_pos) = real_chan, chan_map.shape = (4, NCHAN)
rx_to_tx = ca.ib[0].mb.RX_TO_TX_LANE_MAP # Get the backplane lane map
chan_map = np.array([[c + 8*rx_to_tx[(rx_slot, rx_lane)][0] for rx_lane in range(4) for c in range(8)] for rx_slot in range(4)])

# This computes the scrambled FFT outputs with bin rotation, 1st-level corner-turn, and bin culling. 
# bins_culled = np.array([[2048*((lane - (clk & 0b11)) & 0b11) + clk for clk in range(2048) if not (clk < 256 and clk & 0b11 == lane)] for lane in range(4)]) 

# Apply correlator bin decimation
# corr_bins := (slot, bins) = bin   corr_bins.shape=(4, 496) (496 = 1984/4)
corr_bins = ct2_bins[:, 3::4]

# Compute (i,j) product indices coming out of the correlator on each clock for a single bin 
# The map reflects the way the correlator performs the products 
# prod_map := (clk, prod) = (i, j)   prod_map.shape=(4, 132, (2))
N=32
NCLK=4
prod_map=np.array([[(i, (j+i) % N) if i < N-j else ((j+i) % N, i-1) for i in range(N+1) for j in range(N//2//NCLK*clk, N//2//NCLK*(clk+1)) ] for clk in range(NCLK) ])

# Compute (i.j) product indices for each slot, taking account the slot-dependent channel scrambling 
# prod_remap := (slot, clk, prod) = (i,j)  prod_remap.shape = (4, 4, 132, (2))
prod_remap = np.array([slot_chan_map[prod_map] for slot_chan_map in chan_map])

# We now have all the elements to create a map that described the bin number  and (i,j) product indices 
# for each of the values received in the packet receiver.
# The receiver stores data as (slot_id, clkbin_id, prod) .shape=(4,496,132). 
# The map profides a (bin, i, j) tuple for each of the element of that array 
# nmap := (slot_id, clkbin_id, prod) = (bin, i, j) nmap.shape = (4, 1984, 132, (3)), 1984=4 clk * 496 bins, bin=0-8191, i,j=0-NCHAN-1 
nmap =  np.array([[[(bin_, i, j) for i,j in clkprod]  for bin_ in bins_culled[slot] for clkprod in prod_remap[slot] ] for slot in range(4)])



In [ ]:
nmap.shape

In [ ]:
ct1_bins[0, :500]

In [ ]:
bins_culled.shape # (slot, bin_id) = bin

In [ ]:
bins_culled[:, 3::4]

In [ ]:
# We now want a remap array (slot_id, clkbin_id, prod) (4,512,132) into (bin, i, j) (8192, 32, 32) 
nmap =  np.array([[[(bin_, i, j) for i,j in clkprod]  for bin_ in bins_culled[slot] for clkprod in prod_remap[slot] ] for slot in range(4)])

In [ ]:
nmap.shape #(slot, clkbin_id, prod) = (bin, i, j)

In [ ]:
nmap[0,0,:100]

In [ ]:
a=np.zeros((4,8192,132,2), dtype=np.int32)

In [ ]:
i1=np.zeros((8192,1024), dtype=np.int32)

In [ ]:
b=a[i1,i1,i1]

In [ ]:
b.shape

In [ ]:
b=i1

In [ ]:
b[a,a]=a

In [ ]:
b.shape

In [ ]:
a=np.array([[1,2],[3,4]])

In [ ]:
a[[0,1,1],[1,0,0]]=[4,5,6]

In [ ]:
a

In [ ]:
m=np.array(prod_map)

In [ ]:
132*5+10

In [ ]:
chan_map

In [ ]:
rx_to_tx

In [ ]:
m=np.array(chan_map)


In [ ]:
chan_map[0, prod_map]

In [ ]:
m.shape

In [ ]:
prod_remap = np.array([chan_map[i,slot_map] for slot_map in prod_map])

In [ ]:
prod_map[0], p[0]

In [ ]:
[(x,y,z) for x in "ab" for y in "cd" for z in "ef"]

In [ ]:
print("hello world")


In [ ]:
class Person:
    age = 0
    name = "nobody"

In [ ]:
p1 = Person()

In [ ]:
p1.age=2

In [ ]:
4*(11,22)

In [ ]:
type(int)

In [ ]:
a=np.array([1<<10,2], dtype=np.float16)

In [ ]:
a.tobytes().hex(':')

In [ ]:
a=np.array([1,2,3,4])

In [ ]:
b=np.array([[0, 1],[2,3]])

In [ ]:
a

In [ ]:
b

In [ ]:
a[b]

In [ ]:
a=np.array([[4,5,6,7],[6,7,8,9], [1,2,3,4], [4,4,4,4]])

In [ ]:
a

In [ ]:
a.shape

In [ ]:
a[1]

In [ ]:
a[b,b]

In [ ]:
a=np.empty((5,4,3))

In [ ]:
a[[0,1],[2,3]]=b

In [ ]:
x=np.arange(2)

In [ ]:
a[x,x]=b

In [ ]:
a[x[None,:],x[:, None],x]=b

In [ ]:
np.take(a, axis=1)

In [ ]:
b[0,:][None,:]

In [ ]:
np.reshape(b,(None,1))

In [ ]:
dict(zip((1,2,3),[4,5,6]))

In [ ]:
i=ca.ib[1]

In [ ]:
i.set_user_output_source(source='bp_trig',output=0)

In [ ]:
i.set_user_output_source?

In [ ]:
for i in ca.ib:
    i.set_irigb_source_sync('~bp_trig')

for i in ca.ib:
    print(f'{i}: {i.get_irigb_time_sync()}')

In [ ]:
a=0xFFFF
hex(a & ~(1<<15))

In [ ]:
ca.print_iceboard_info()

In [ ]:
i=ca.ib[0]

In [ ]:
ca.ib.get_temperatures()

In [ ]:
i.i2c0_bp_tmp421.get_temperature()

In [ ]:
sensors = {'PLL': 'i2c0_tmp421_pll', '0.85V':'i2c1_tmp422_0v85', '1.2Va':'i2c1_tmp421_1v2a', '1.2Vb':'i2c1_tmp421_1v2b', '1.4V':'i2c1_tmp421_1v4', '1.8V':'i2c1_tmp421_1v8', '2.5V': 'i2c1_tmp421_2v5', '3.3V': 'i2c1_tmp421_3v3', '5.0V':'i2c1_tmp421_5v0'}
table = []
rows= ['FPGA'] + list(sensors)
cols = [f'{ib!r} Slot {ib.slot}' for ib in ca.ib]
for ib in ca.ib:
    col=[]
    fpga_temp = ib.SYSMON.temperature()
    col.append(f'{fpga_temp:4.1f}')
    for name, sensor in sensors.items():
        temps = getattr(ib, sensor).get_temperature() 
        col.append(' '.join(f'Ext{p} {t:4.1f}' if p else f'Int {t:4.1f}' for p,t in enumerate(temps[:-1])))
    table.append(col)
ca.print_table(table, row_labels=rows, col_labels=cols)

In [ ]:
    def print_iceboard_temperatures(self):
        sensor = self.ib[0].TEMPERATURE_SENSOR.MB_FPGA_DIE
        self.print_iceboard_table(
            lambda ib: '%3.1f' % ib.get_motherboard_temperature(sensor),
            row_labels='FPGA Die Temp')


In [ ]:
i.SYSMON.temperature()

In [ ]:
await i.fpga_core_reg_write_async(15*4, 1<<28)

In [ ]:
for i in ca.ib:
    print(f'{await i.fpga_core_reg_read_async(15*4):08x}')

In [ ]:
i= ca.ib[0]
for _ in range(100):
    print(f'{i}: {i.get_irigb_time_sync(trig=False)}')

In [ ]:
bin(~((1<<28) -1))

In [ ]:
4.39804651e+12/2**31

In [ ]:
a = 1
b = -8276338268327683

a1= a * b & 0x3FFFF
a2 = (a* (b>>18))
a3 = a1 + (a2<<18)
print(f'{a1=}, {a2=}, {a3=} Expected {a*b}, {("ERROR","OK")[a3==a*b]}')

In [ ]:
a2

In [ ]:
bin(a1)

In [ ]:
bin(a2<<2)

In [ ]:
a1|(a2<<18)

In [ ]:
a2 << 18

In [ ]:
a1

In [ ]:
a3

In [ ]:
bin(-65535 & 0xFFFFFFFFFFFF)

In [ ]:
(-200000 * 17000) & 0xFFFFFFFF

In [ ]:
# BER tester

In [ ]:
for i in ca.ib:
    i.GPIO.ANT_RESET=1
    i.GPIO.CORR_RESET=1
    i.GPIO.CORR_RESET=0
    i.GPIO.ANT_RESET=0

# Set all transmitters
for ib in ca.ib:
    for yy in ib.CT.BPLINKS.gty:
        yy.TXPRBSSEL = 0b0101
        yy.RXPRBSSEL = 0b0101
        yy.TXDIFFCTRL = 15
        yy.TXPRECURSOR = 0
        yy.TXPOSTCURSOR = 0
        yy.TXPRBSFORCEERR = 0
        yy.RXLPMEN=0
        yy.TXINHIBIT=0
        yy.LOOPBACK=0

# Retrain all the Rx equalizers
for ib in ca.ib:
    for yy in ib.CT.BPLINKS.gty:
        yy.reset_rx_equalizer()

# Clear error counters for all links
for ib in ca.ib:
    for yy in ib.CT.BPLINKS.gty:
        yy.RXPRBSCNTRESET=1
        yy.RXPRBSCNTRESET=0

# Wait for the errors to accumulate
period = 0.1
time.sleep(period)

# Gather the error counts
for ib in ca.ib:
    for yy in ib.CT.BPLINKS.gty:
        print(f'{yy}: err={yy.ERR_CTR}, BER < {(yy.ERR_CTR or 1)/(25e9*period):5.1e}')


In [ ]:
yy=i.CT.BPLINKS.gty[0]

In [ ]:
i.CT.BPLINKS.RX_RESET

In [ ]:
yy.RXPRBSCNTRESET=0
yy.RXPRBSCNTRESET=1
yy.ERR_CTR

In [ ]:
hex(335544320)

In [ ]:
yy.RXRESETDONE

In [ ]:
i.sync()